# Threshold memristor pulse protocol

This notebook runs a 24-case amplitude/duration/starting-state matrix, a timestep-refinement study, and one held-out stimulus using Kessetsu's exact bundled threshold-model dependency. It is a synthetic model-characterization workflow, not a physical-device fit or mechanism validation.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from kessetsu import KessetsuClient

repo = Path.cwd()
examples = repo / 'examples'
if not (examples / 'memristor_pulse_protocol.kessstudy.json').exists():
    raise RuntimeError('Run this notebook from the Kessetsu repository or release-bundle root')
artifacts = repo / '.artifacts' / 'memristor-protocol-notebook'
artifacts.mkdir(parents=True, exist_ok=True)
client = KessetsuClient()
print(f'Kessetsu CLI {client.version()}')

In [ ]:
def run(name, output):
    target = artifacts / output
    if target.exists():
        target.unlink()
    return client.run_study(examples / name, target)

matrix = run('memristor_pulse_protocol.kessstudy.json', 'matrix.json')
convergence = run('memristor_convergence.kessstudy.json', 'convergence.json')
holdout = run('memristor_holdout.kessstudy.json', 'holdout.json')
print(matrix.raw['summary'], convergence.raw['summary'], holdout.raw['summary'])

In [ ]:
def seconds(text):
    for suffix, scale in [('ps', 1e-12), ('ns', 1e-9), ('s', 1.0)]:
        if text.endswith(suffix):
            return float(text[:-len(suffix)]) * scale
    raise ValueError(f'Unsupported duration {text}')

def ohms(text):
    return float(text.removesuffix('kOhm')) * 1e3 if text.endswith('kOhm') else float(text.removesuffix('Ohm'))

def effective_state(study, case):
    duration = seconds(case['parameters']['pulse_duration'])
    frame = study.simulation(case['id']).table().to_pandas()
    def read(start, stop):
        window = frame[(frame['time'] >= start) & (frame['time'] <= stop)].copy()
        window = window[window['v_vsense#branch'].abs() > 1e-12]
        return (window['top'] / window['v_vsense#branch']).median()
    return read(0.35e-9, 0.65e-9), read(duration + 2.15e-9, duration + 2.45e-9)

def protocol_table(study):
    measured = study.measurements_table().to_pandas().set_index('case_id')
    rows = []
    for case in study.raw['plan']['cases']:
        before, after = effective_state(study, case)
        row = measured.loc[case['id']]
        rows.append({**case['parameters'], 'case_id': case['id'], 'before_ohm': before, 'after_ohm': after, 'change_ohm': after-before, 'energy_joule': row['measurement.protocol_energy'], 'peak_current_ampere': row['measurement.peak_current']})
    return pd.DataFrame(rows)

matrix_table = protocol_table(matrix)
matrix_table

In [ ]:
matrix_table['amplitude_v'] = matrix_table['pulse_amplitude'].str.removesuffix('V').astype(float)
matrix_table['duration_ns'] = matrix_table['pulse_duration'].map(seconds) * 1e9
matrix_table['initial_kohm'] = matrix_table['initial_resistance'].map(ohms) / 1e3
fig, (state_axis, energy_axis) = plt.subplots(1, 2, figsize=(11, 4.5))
for (initial, duration), group in matrix_table.groupby(['initial_kohm', 'duration_ns']):
    ordered = group.sort_values('amplitude_v')
    state_axis.plot(ordered['amplitude_v'], ordered['change_ohm'] / 1e3, 'o-', label=f'{initial:g} kΩ, {duration:g} ns')
state_axis.axvspan(-1.6, 1.6, color='0.5', alpha=0.12, label='Below threshold')
state_axis.axhline(0, color='0.5', linewidth=1)
state_axis.set(xlabel='Write amplitude (V)', ylabel='Modeled resistance change (kΩ)', title='Signed pulse response')
state_axis.grid(alpha=0.25)
state_axis.legend(fontsize=8, ncol=2)
for initial, group in matrix_table.groupby('initial_kohm'):
    energy_axis.scatter(group['duration_ns'], group['energy_joule'], label=f'{initial:g} kΩ start')
energy_axis.set(xlabel='Write duration (ns)', ylabel='Protocol energy (J)', yscale='log', title='Finite positive simulated energy')
energy_axis.grid(alpha=0.25)
energy_axis.legend()
fig.tight_layout()
fig.savefig(artifacts / 'pulse-protocol.png', dpi=160)
fig

In [ ]:
convergence_table = protocol_table(convergence)
convergence_table['timestep_ps'] = convergence_table['timestep'].map(seconds) * 1e12
keys = ['pulse_amplitude', 'pulse_duration', 'initial_resistance']
coarse = convergence_table[convergence_table['timestep_ps'].round(6) == 25.0].set_index(keys)
fine = convergence_table[convergence_table['timestep_ps'].round(6) == 12.5].set_index(keys)
sensitivity = pd.DataFrame({
    'state_relative_difference': (coarse['after_ohm'] - fine['after_ohm']).abs() / pd.concat([coarse['after_ohm'].abs(), fine['after_ohm'].abs()], axis=1).max(axis=1),
    'energy_relative_difference': (coarse['energy_joule'] - fine['energy_joule']).abs() / pd.concat([coarse['energy_joule'].abs(), fine['energy_joule'].abs()], axis=1).max(axis=1),
}).reset_index()
print('Maximum final-state difference:', f"{100*sensitivity['state_relative_difference'].max():.3f}%")
print('Maximum energy difference:', f"{100*sensitivity['energy_relative_difference'].max():.3f}%")
sensitivity

In [ ]:
holdout_table = protocol_table(holdout)
print('Held-out stimulus (not used to choose the sweep): 2.4 V, 4 ns, 5 kΩ start')
holdout_table

The output directory retains all three versioned study artifacts and the plotted summary. The pre/post resistance is an explicit Ohm's-law readback from the two 0.4 V windows, not a hidden simulator state or a new Core measurement. The held-out point checks the protocol away from the displayed sweep grid, but it is still generated by the same artificial model and Ngspice solver; independent measurements would be required for physical validation.